## Project 2: Tidying and Transforming Wide Datasets 
## IS 362 Data Acquisition and Management

## Dataset 1: Book Sales
This dataset tracks book sales over a three month period. It is currently in a wide format because the months (January Sales, February Sales, March Sales) are listed as individual columns rather than data values under a single Month variable.

We will  melt the month columns into a single Month column, store the numeric values in a Sales_Count column, and take away the sales from the end of the months to debloat the data

In [ ]:
!pip install pandas


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [23]:
import pandas as pd

In [ ]:
# getting the data
df_books = pd.read_csv('books.csv')
print("--- Original Bookstore Data ---")
display(df_books)

# tidying the dataset 
tidy_books = df_books.melt(
    id_vars=['Book Title', 'Genre'], 
    value_vars=['January Sales', 'February Sales', 'March Sales'],
    var_name='Month', 
    value_name='Sales_Count'
)

# debloating the data
tidy_books['Month'] = tidy_books['Month'].str.replace(' Sales', '')

print("\n--- Cleaned bookstore Data ---")
display(tidy_books.head())

# total sales by genre
print("\n--- Top selling books ---")
top_books = tidy_books.groupby('Book Title')['Sales_Count'].sum().reset_index()
display(top_books.sort_values('Sales_Count', ascending=False))
# top selling genres
print("\n--- Top selling genres ---")
top_genres = tidy_books.groupby('Genre')['Sales_Count'].sum().reset_index()
display(top_genres.sort_values('Sales_Count', ascending=False))
# montly sales
print("\n--- Monlthy sales count ---")
monthly = tidy_books.groupby('Month')['Sales_Count'].sum().reset_index()
month_dict = {'January': 1, 'February': 2, 'March': 3}
monthly['Sort_Num'] = monthly['Month'].map(month_dict)
monthly = monthly.sort_values('Sort_Num').drop(columns=['Sort_Num'])
display(monthly) 
# romance comparison 
print("\n--- Romance books comparison over time ---")
romance = tidy_books[tidy_books['Genre'] == 'Romance']
romance_pivot = romance.pivot(index='Month', columns='Book Title', values='Sales_Count')
romance_pivot.index = pd.Categorical(romance_pivot.index, categories=['January', 'February', 'March'], ordered=True)
display(romance_pivot.sort_index())

--- Original Bookstore Data ---


,Book Title,Genre,January Sales,February Sales,March Sales
0,The Last Summer,Romance,45,52,61
1,Midnight House,Horror,32,41,38
2,Lost in the Stars,Science Fiction,27,35,46
3,The Silent Case,Mystery,51,48,57
4,Second Chances,Romance,39,44,50



--- Cleaned bookstore Data ---


,Book Title,Genre,Month,Sales_Count
0,The Last Summer,Romance,January,45
1,Midnight House,Horror,January,32
2,Lost in the Stars,Science Fiction,January,27
3,The Silent Case,Mystery,January,51
4,Second Chances,Romance,January,39



--- Top selling books ---


...

,Genre,Sales_Count
2,Romance,291
1,Mystery,156
0,Horror,111
3,Science Fiction,108



--- Monlthy sales count ---


,Month,Sales_Count
1,January,194
0,February,220
2,March,252



--- Romance books comparison over time ---


Book Title,Second Chances,The Last Summer
January,39,45
February,44,52
March,50,61


Bookstore Analysis Conclusion:
The clean data allows us to easily look through the sales figures. Overall, The Last Summer was the best selling individual book, driving Romance to be the most popular genre overall. The store saw a steady month over month increase in total sales, growing from 194 units in January to 252 units in March. When comparing the two romance novels directly, The Last Summer consistently outsold Second Chances every single month.

## Dataset 2: Brooklyn Demographics
This dataset tracks Brooklyn's population across 1920, 1970, and 2010. its a bit untidy because it mixes absolute population counts and calculated percentages horizontally across the row for every demographic group, which can be too much to look at.

isolaitng the absolute population counts and melt them into a Demographic_Group column and a Count column, dropping historical null values where certain demographics were not tracked.

In [25]:
# loading the data
df_demo = pd.read_csv('demographics.csv')
df_demo['Borough'] = 'Brooklyn'
print("--- Original Demographic Data ---")
display(df_demo)

# tidying the data: selecting only the count columns to melt
count_columns = ['White', 'Non-Hispanic Whites', 'Black', 'Asian', 'Other or Mixed', 'Hispanic/Latino']

tidy_demo = df_demo.melt(
    id_vars=['Borough', 'Year', 'Population'], 
    value_vars=count_columns,
    var_name='Demographic_Group', 
    value_name='Count'
)

# dropping the NaN values to convert to integer values
tidy_demo = tidy_demo.dropna(subset=['Count'])
tidy_demo['Count'] = tidy_demo['Count'].astype(int)

print("\n--- Tidied Demographic Data ---")
display(tidy_demo.sort_values(by=['Year', 'Demographic_Group']).head())

# Analysis: total shift in borough demogrpahic
demo_pivot = tidy_demo.pivot(index='Year', columns='Demographic_Group', values='Count').fillna(0)
display(demo_pivot)

print("\n--- Total shift by demographic group ---")
# calculates the absolute shift (max count minus min count) to see which demographic changed the most
shift = demo_pivot.max() - demo_pivot.min()
shift_df = shift.sort_values(ascending=False).reset_index(name='Total_Shift_Count')
display(shift_df)

--- Original Demographic Data ---


,Year,Population,White,% W,Non-Hispanic Whites,% ANG,Black,% B,Asian,% A,Other or Mixed,% O/M,Hispanic/Latino,% H/L,Borough
0,1920,2018356,1984953,98.35,NaN,NaN,31912,1.58,1459,0.07,32,0.00,NaN,NaN,Brooklyn
1,1970,2602012,1905788,73.24,NaN,NaN,656194,25.22,16138,0.62,23892,0.92,NaN,NaN,Brooklyn
2,2010,2504700,1072041,42.80,893306.0,35.67,860083,34.34,263519,10.52,309057,12.34,496285.0,19.81,Brooklyn



--- Tidied Demographic Data ---


,Borough,Year,Population,Demographic_Group,Count
9,Brooklyn,1920,2018356,Asian,1459
6,Brooklyn,1920,2018356,Black,31912
12,Brooklyn,1920,2018356,Other or Mixed,32
0,Brooklyn,1920,2018356,White,1984953
10,Brooklyn,1970,2602012,Asian,16138


Demographic_Group,Asian,Black,Hispanic/Latino,Non-Hispanic Whites,Other or Mixed,White
Year,,,,,,
1920,1459.0,31912.0,0.0,0.0,32.0,1984953.0
1970,16138.0,656194.0,0.0,0.0,23892.0,1905788.0
2010,263519.0,860083.0,496285.0,893306.0,309057.0,1072041.0



--- Total shift by demographic group ---


,Demographic_Group,Total_Shift_Count
0,White,912912.0
1,Non-Hispanic Whites,893306.0
2,Black,828171.0
3,Hispanic/Latino,496285.0
4,Other or Mixed,309025.0
5,Asian,262060.0


Demographic shift  analysis conclusion:
By melting the dataset and removing the initial N/A values, it isolates the pure population counts for each demographic group. Because categories like Hispanic/Latino and Non-Hispanic Whites were not tracked in the 1920 and 1970 census years, filled those missing historical data points with zeros in the table to accurately measure their emergence. By calculating the absolute difference between the maximum and minimum population counts across the century,it reveals that the White demographic experienced the largest overall change shifting by over 912,000 residents, closely followed by the Black demographic, which saw the largest pure growth, increasing by over 828,000 residents.

## Dataset 3: GPU Benchmarks
This hardware dataset compares three graphics cards. its untidy because the column headers (1080p Medium (FPS), 1440p Ultra (FPS),) are specific graphical settings containing performance metrics, rather than distinct variables.

melting the resolution settings into a Resolution_Setting column and map the intersecting numbers into a designated Average_FPS column.

In [26]:
# loading the data
df_gpu = pd.read_csv('gpus.csv')
print("--- Original GPU Data ---")
display(df_gpu)

# tidying the data
tidy_gpu = df_gpu.melt(
    id_vars=['GPU Model'], 
    var_name='Resolution_Setting', 
    value_name='Average_FPS'
)

print("\n--- Cleaned GPU data ---")
display(tidy_gpu.head())

# finding the performance drop off for each GPU
gpu_performance = tidy_gpu.groupby('GPU Model')['Average_FPS'].agg(
    Max_FPS='max', 
    Min_FPS='min'
).reset_index()

# calculateing the drop from the easiest to hardest setting
gpu_performance['FPS_Drop'] = gpu_performance['Max_FPS'] - gpu_performance['Min_FPS']

print("\n--- Performance Drop off (1080p Medium to 4K Ultra) ---")
display(gpu_performance.sort_values(by='FPS_Drop', ascending=False))

--- Original GPU Data ---


,GPU Model,1080p Medium (FPS),1080p Ultra (FPS),1440p Ultra (FPS),4K Ultra (FPS)
0,Nvidia GeForce RTX 4090,258.9,225.5,196.4,136.2
1,AMD Radeon RX 7900 XTX,253.9,215.1,179.6,110.4
2,Nvidia GeForce RTX 3080,179.3,144.5,111.4,66.8



--- Cleaned GPU data ---


,GPU Model,Resolution_Setting,Average_FPS
0,Nvidia GeForce RTX 4090,1080p Medium (FPS),258.9
1,AMD Radeon RX 7900 XTX,1080p Medium (FPS),253.9
2,Nvidia GeForce RTX 3080,1080p Medium (FPS),179.3
3,Nvidia GeForce RTX 4090,1080p Ultra (FPS),225.5
4,AMD Radeon RX 7900 XTX,1080p Ultra (FPS),215.1



--- Performance Drop off (1080p Medium to 4K Ultra) ---


,GPU Model,Max_FPS,Min_FPS,FPS_Drop
0,AMD Radeon RX 7900 XTX,253.9,110.4,143.5
2,Nvidia GeForce RTX 4090,258.9,136.2,122.7
1,Nvidia GeForce RTX 3080,179.3,66.8,112.5


GPU benchmarks result:
melting the resolutions into a single column, each row now represents one distinct pairing (one GPU at one specific resolution). Grouping by the GPU Model allows us to calculate the absolute FPS drop off between the least demanding setting and the most demanding setting. The final result reveals that while the RTX 4090 outputs the highest overall frames, it also experiences the largest absolute drop off (122.7 FPS) when resolution is altered or maxamized.